# Fraud Detection — Vertex AI Interactive Inference

Calls the deployed XGBoost fraud model on Vertex AI and returns fraud scores with SHAP explanations.

**Prerequisites**  
Run `deploy_to_vertex.py` first, which writes `vertex_config.json` with your endpoint ID.

In [ ]:
# pip install google-cloud-aiplatform pandas --quiet

## 1. Connect to Vertex AI Endpoint

In [ ]:
import json
from pathlib import Path
from google.cloud import aiplatform

# Load connection details written by deploy_to_vertex.py
config = json.loads(Path("vertex_config.json").read_text())

PROJECT_ID   = config["project_id"]
REGION       = config["region"]
ENDPOINT_ID  = config["endpoint_id"]

aiplatform.init(project=PROJECT_ID, location=REGION)
endpoint = aiplatform.Endpoint(ENDPOINT_ID)

print(f"Connected to endpoint: {ENDPOINT_ID}")

## 2. Helper — Build a Request Payload

In [ ]:
def make_payload(
    feature_vectors: list[list[float]],
    explain: bool = True,
    top_k_features: int = 5,
) -> bytes:
    """Encode a list of feature vectors into the API's BatchRequest format."""
    payload = {
        "transactions": [{"features": fv} for fv in feature_vectors],
        "explain": explain,
        "top_k_features": top_k_features,
    }
    return json.dumps(payload).encode()


def score(feature_vectors: list[list[float]], **kwargs):
    """Score one or more transactions and return parsed predictions."""
    response = endpoint.raw_predict(
        body=make_payload(feature_vectors, **kwargs),
        headers={"Content-Type": "application/json"},
    )
    return json.loads(response.data)["predictions"]

## 3. Score a Single Transaction

In [ ]:
# Feature order matches the model training schema.
# Edit these values to test different transactions.
single_transaction = [
    125.50,   # txn_amount
    4.83,     # txn_amount_log
    0.70,     # merchant_risk_score
    2.50,     # hours_since_last_txn
    3.0,      # txn_count_1h
    12.0,     # txn_count_24h
    98.00,    # avg_amount_7d
    45.20,    # std_amount_7d
    850.0,    # distance_from_home
    0.0,      # is_international
]

predictions = score([single_transaction])
pred = predictions[0]

print(f"Fraud probability : {pred['fraud_probability']:.4f}")
print(f"Classification    : {'FRAUD' if pred['is_fraud'] else 'LEGIT'} (threshold={pred['threshold']})")

if pred.get("top_features"):
    print("\nTop SHAP features:")
    for feat in pred["top_features"]:
        arrow = "▲" if feat["direction"] == "increases_risk" else "▼"
        print(f"  {arrow} {feat['feature']:30s}  value={feat['value']:8.3f}  shap={feat['shap_value']:+.4f}")

## 4. Batch Score Multiple Transactions

In [ ]:
import pandas as pd

transactions = [
    [125.50, 4.83, 0.70, 2.50,  3, 12,  98.00, 45.20,  850.0, 0.0],  # normal
    [4200.0, 8.34, 0.95, 0.08, 12, 45, 120.00, 80.50, 3200.0, 1.0],  # suspicious
    [32.00,  3.47, 0.10, 18.0,  1,  4,  35.00, 10.20,    5.0, 0.0],  # normal
    [9999.0, 9.21, 0.99, 0.02, 20, 80, 200.00, 95.00, 5000.0, 1.0],  # high risk
]

predictions = score(transactions, explain=False)  # skip SHAP for speed

rows = []
for i, pred in enumerate(predictions):
    rows.append({
        "txn": i,
        "fraud_prob": pred["fraud_probability"],
        "is_fraud": pred["is_fraud"],
    })

pd.DataFrame(rows).style.background_gradient(subset=["fraud_prob"], cmap="RdYlGn_r")

## 5. Score From a DataFrame

In [ ]:
# Plug in your own DataFrame here.
# df must have columns in the same order as the model's feature schema.

FEATURE_COLS = [
    "txn_amount", "txn_amount_log", "merchant_risk_score",
    "hours_since_last_txn", "txn_count_1h", "txn_count_24h",
    "avg_amount_7d", "std_amount_7d", "distance_from_home", "is_international",
]

# df = pd.read_csv("your_data.csv")
# preds = score(df[FEATURE_COLS].values.tolist(), explain=False)
# df["fraud_probability"] = [p["fraud_probability"] for p in preds]
# df["is_fraud"] = [p["is_fraud"] for p in preds]
# df.head()

## 6. Adjust the Decision Threshold

The model returns raw probabilities — you can apply any threshold without redeploying.

In [ ]:
CUSTOM_THRESHOLD = 0.15  # lower = more sensitive, more false positives

predictions = score(transactions, explain=False)
for i, pred in enumerate(predictions):
    prob = pred["fraud_probability"]
    flag = prob >= CUSTOM_THRESHOLD
    print(f"txn {i}: prob={prob:.4f}  flag={flag}")